# FP net test

 - Loads the floating point network
 - Runs on the validation dataset 
 - Check the accuracy

To check that the accuracy is close to the one obtained with snn torch

In [1]:
from pathlib import Path
import numpy as np
from lava_network.spiking_dataloader import WISDM_spiking_dataloader, WisdmDatasetParser
from lava_network.output_process import OutputProcess

In [2]:
import json
def deserialize_dict(json_str):
    def convert(obj):
        if isinstance(obj, list):
            return np.array(obj).astype(np.int64)
        if isinstance(obj, int):
            return np.int64(obj)
        return obj

    return json.loads(json_str, object_hook=lambda d: {k: convert(v) for k, v in d.items()})

with open('network_fixed.json', 'r') as json_file:
    loaded_json_str = json_file.read()
converted_params = deserialize_dict(loaded_json_str)


In [3]:
total_sample = 6000
signal_step = 40
clear_intervall = 4
time_steps = signal_step + clear_intervall

#high score split [0, 2, 6, 8, 9, 14, 17]
#worst score split [0,1,2,5,6,7,12]
# balanced split [0, 1, 4, 8, 9, 10, 14]
dataset = WisdmDatasetParser('../data/data_watch_subset_0_40.npz', norm=None, class_sublset='custom', subset_list=[0, 1, 4, 8, 9, 10, 14])
val_set = dataset.get_validation_set(shuffle=False, subset=None)
#val_set = dataset.get_validation_set()
# train_set = (val_set[0][:int(train_percentage*val_set[0].shape[0])], val_set[1][:int(train_percentage*val_set[0].shape[0])])
# val_set = (val_set[0][int(train_percentage*val_set[0].shape[0]):], val_set[1][int(train_percentage*val_set[0].shape[0]):])

num_samples = val_set[0].shape[0]
spiking_loader = WISDM_spiking_dataloader(val_set ,clear_intervall=clear_intervall)
out_sink = OutputProcess(7,num_samples,time_steps, 0)

(6,)
(6,)
ytrain shape (55404, 18)
yval shape (18468, 18)
ytest shape (18469, 18)
num classes train dataset: 7 occurrences of each class:[3127 3066 3044 3047 3150 3087 2973]
num classes eval dataset: 7 occurrences of each class:[1035  968 1048  996 1110 1053 1007]
num classes test dataset: 7 occurrences of each class:[1046 1061 1048 1036 1076 1026  982]


In [4]:
from lava.proc.lif.process import LIF
from lava.proc.dense.process import Dense
from lava.utils.weightutils import SignMode

#path = f"{Path.home()}/snntorch_network/notebook/Trained/network_best.npz"
#path = f"{Path.home()}/snntorch_network/nni_experiments/inibitory_lif_no_encoder_worst/results/ipk2erm5/trials/whXvC/Trained/network_best.npz"
# path = f"{Path.home()}/snntorch_network/nni_experiments/inibitory_lif_no_encoder_balanced/results/4m8j0yfa/trials/yAnF7/Trained/network_best.npz"
#path = f"{Path.home()}/snntorch_network/nni_experiments/inibitory_lif_no_encoder_best/results/ot6eqima/trials/oqxoS/Trained/network_best.npz"
path = "../data/inibitory_lif_no_encoder_balanced.npz"

data = np.load(path,allow_pickle=True)


linear1_w= data['linear1']
leaky1_vth= data['leaky1_vth']
leaky1_betas= 1-data['leaky1_betas']
leaky1_betas= leaky1_betas if leaky1_betas >= 0 else np.zeros(leaky1_betas.shape)
print(f"leaky1_betas: {leaky1_betas}")
print(f"leaky1_vth: {leaky1_vth}")
linear2_w = data['linear2']
leaky2_vth= data['recurrent_vth']
leaky2_betas= 1 - data['recurrent_betas']
leaky2_betas= leaky2_betas if  leaky2_betas >= 0 else np.zeros(leaky2_betas.shape)
print(f"leaky2_betas: {leaky2_betas}")
print(f"leaky2_vth: {leaky2_vth}")

recurrent_in_weights = data['input_dense']
recurrent_out_weights = - data['output_dense']
recurrent_vth = data['activation_vth']
recurrent_leaky_betas = 1 - data['activation_betas']
recurrent_leaky_betas= recurrent_leaky_betas if recurrent_leaky_betas >= 0 else np.zeros(recurrent_leaky_betas.shape)
print(f"recurrent_leaky_betas: {recurrent_leaky_betas}")
print(f"recurrent_vth: {recurrent_vth}")

linear3_w = data['linear3']
leaky3_vth= data['leaky2_vth']
leaky3_betas= 1 - data['leaky2_betas']
leaky3_betas= leaky3_betas if leaky3_betas >= 0 else np.zeros(leaky3_betas.shape)
print(f"leaky3_betas: {leaky3_betas}")
print(f"leaky3_vth: {leaky3_vth}")


# Block structure of the AHPC network
#       +-----------+     +-----------+    +-----------+              +-----------+          +-----------+     +-----------+
# +--+  |           |     |           |    |           |  +--+        |           |          |           |     |           |    +--+
# |i |->|  In_Dense |---->|  In_LIF   |--->|  In_Dense |--| -|------->|  F_LIF    |-------+->| Out_Dense |---->|  O_LIF    |--->|o |
# +--+  |           |     |           |    |           |  +--+        |           |       |  |           |     |           |    +--+
#       +-----------+     +-----------+    +-----------+    A         +-----------+       |  +-----------+     +-----------+
#                                                           |                             |
#                                                           |  +------+ +------+ +------+ |
#                                                           |  |      | |      | |      | |
#                                                           +--|Dense |<| ILIF |<|Dense |<+
#                                                              |      | |      | |      |
#                                                              +------+ +------+ +------+
#



linear1 = Dense(weights=linear1_w, num_message_bits=32, name="linear1")

leaky1 = LIF(shape=(linear1_w.shape[0],),
                    u = np.zeros(linear1_w.shape[0]),
                    v = np.zeros(linear1_w.shape[0]),
                    du = 1.0,
                    dv = leaky1_betas,
                    vth=leaky1_vth,
                    log_config=0,
                    name= "leaky1"
                )
linear1.a_out.connect(leaky1.a_in)
linear2 = Dense(weights=linear2_w, num_message_bits=0, sign_mode=SignMode.MIXED, name="linear2")
linear2.s_in.connect_from(leaky1.s_out)

leaky2 = LIF(shape=(linear2_w.shape[0],),
                    u = np.zeros(linear2_w.shape[0]),
                    v = np.zeros(linear2_w.shape[0]),
                    du = 1.0,
                    dv= leaky2_betas,
                    vth=leaky2_vth,
                    log_config=0,
                    name= "leaky2"
                )
#sum.a_out.connect(leaky2.a_in)
linear2.a_out.connect(leaky2.a_in)
#leaky2.a_in.connect_from(linear2.a_out)

recurrent_in = Dense(weights=recurrent_in_weights, num_message_bits=0, name="recurrent_in")
leaky2.s_out.connect(recurrent_in.s_in)

ahpc = LIF(shape=(recurrent_in_weights.shape[0],),
                    u = np.zeros(recurrent_in_weights.shape[0]),
                    v = np.zeros(recurrent_in_weights.shape[0]),
                    du = 1.0,
                    dv = recurrent_leaky_betas,
                    vth=recurrent_vth,
                    log_config=0,
                    name= "inibitory_leaky"
                )

recurrent_in.a_out.connect(ahpc.a_in)

recurrent_out = Dense(weights=recurrent_out_weights, num_message_bits=0, name="recurrent_out")
recurrent_out.s_in.connect_from(ahpc.s_out)
recurrent_out.a_out.connect(leaky2.a_in)

linear3 = Dense(weights=linear3_w, num_message_bits=0, name="linear3")
linear3.s_in.connect_from(leaky2.s_out)

leaky3 = LIF(shape=(linear3_w.shape[0],),
                    u = np.zeros(linear3_w.shape[0]),
                    v = np.zeros(linear3_w.shape[0]),
                    du = 1.0,
                    dv = leaky3_betas,
                    vth=leaky3_vth,
                    log_config=0,
                    name= "leaky3"
                )
leaky3.a_in.connect_from(linear3.a_out)
spiking_loader.data_out.connect(linear1.s_in)
leaky3.s_out.connect(out_sink.spikes_in)
out_sink.label_in.connect_from(spiking_loader.label_out)

leaky1_betas: 1.01234897878021
leaky1_vth: 2.195924758911133
leaky2_betas: 0.07083219289779663
leaky2_vth: 0.8788766860961914
recurrent_leaky_betas: 0.11015737056732178
recurrent_vth: 1.8151839971542358
leaky3_betas: 0.0
leaky3_vth: 0.830508828163147


In [5]:

from lava.magma.core.run_conditions import RunSteps
from lava.magma.core.run_configs import Loihi1SimCfg, Loihi2SimCfg
import numpy as np
from tqdm import tqdm

clock = tqdm(range(num_samples))
for _, i in enumerate(clock):
      out_sink.run(condition=RunSteps(num_steps=time_steps),
                  run_cfg=Loihi1SimCfg(select_sub_proc_model=True,
                  select_tag='floating_pt'))

      leaky1.v.set(np.zeros(leaky1.v.shape))
      leaky1.u.set(np.zeros(leaky1.u.shape))
      leaky2.v.set(np.zeros(leaky2.v.shape))
      leaky2.u.set(np.zeros(leaky2.u.shape))
      leaky3.v.set(np.zeros(leaky3.v.shape))
      leaky3.u.set(np.zeros(leaky3.u.shape))
      ahpc.v.set(np.zeros(ahpc.v.shape))
      ahpc.u.set(np.zeros(ahpc.u.shape))
      linear1.a_buff.set(np.zeros(linear1.a_buff.shape))
      linear2.a_buff.set(np.zeros(linear2.a_buff.shape))
      linear3.a_buff.set(np.zeros(linear3.a_buff.shape))
      recurrent_in.a_buff.set(np.zeros(recurrent_in.a_buff.shape))
      recurrent_out.a_buff.set(np.zeros(recurrent_out.a_buff.shape))

      tmp_predicition = out_sink.pred_labels.get().astype(int)
      current_accuracy = np.sum((tmp_predicition[:i] == val_set[1][:i])/(i+1))*100
      clock.set_description(f"Current accuracy: {current_accuracy:.4f}")
      updated_weights = linear3.weights.get()

# #weight_update_history = monitor_extractor(weight_update)
# pre_trace_value = monitor_extractor(pre_trace_mon)
# post_synaptic_trace_value = np.array(post_synaptic_trace.data.get())
# reward_signal_value = np.array(reward_signal.data.get())
# desired_spikes_value = np.array(desired_spikes.data.get())
# output_value = np.array(out_buffer.data.get())

# Stop the execution
out_sink.stop()


  0%|          | 0/7217 [00:00<?, ?it/s]

Current accuracy: 95.9263: 100%|██████████| 7217/7217 [1:02:08<00:00,  1.94it/s]
